# BLI_segment_sum_contribution: Rf / subgroup contribution 예측

`./data/Cpaper_sample.csv`의 각 행에 대해 `BLI_segment_sum_contribution` 모델로 Rf와 fragment별 contribution을 예측하고 CSV로 저장합니다.

**참고**
- 입력 CSV의 `ratio1`, `ratio2`는 이미 **mol ratio**이고, 모델 학습 데이터(`C_paper.csv`)도 같은 mol ratio를 사용했습니다. analyzer/data manager 안에는 volume→mol 변환 코드가 없으므로 **그대로** 넣습니다.
- 모델은 `ratio1`만 사용합니다: `bulk = ratio1 * elu1 + (1 - ratio1) * elu2`.
- `get_score()`의 `y`와 `subgroup_contributions`는 **표준화된(scaled) 공간**의 값입니다.
  - `y = Σ subgroup_contributions`
  - `Rf = mean + std * y` (= `scaler.inverse_transform(y)`, `predict()` 결과와 동일)
  - 그래서 Rf 단위 기여도를 `contribution_Rf = std * contribution`으로 함께 저장합니다. 이때 `Rf = mean + Σ contribution_Rf`가 성립합니다.
- `subgroup_contributions`는 [-1, 1]로 정규화되어 있지 **않습니다**(샘플에서 1.51, -1.11 등이 나옴). 그래서 `normalized_subgroup_contribution = tanh(contribution)`을 함께 저장합니다.

In [5]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import D4CMPP_TLC
import SSC

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
INPUT_CSV = "./data/Cpaper_sample.csv"
OUTPUT_CSV = "./data/Cpaper_sample_BLI_segment_sum_contribution_pred.csv"
MODEL_NAME = "BLI_segment_sum_contribution"
BATCH_SIZE = 64

In [10]:
BLI_segment = SSC.src.SSCAnalyzer.SSCAnalyzer_v1p3("BLI_segment_sum_contribution")

# sk_scaler = BLI_segment.scaler.scaler  # sklearn StandardScaler
# # RF_MEAN = float(sk_scaler.mean_[0])
# # RF_STD = float(sk_scaler.scale_[0])
# print(f"Rf scaler: mean={RF_MEAN:.6f}, std={RF_STD:.6f}")

founded model: ./Trained_models/BLI_segment_sum_contribution
Device: cuda
#params: 184707
Loading params from ./Trained_models/BLI_segment_sum_contribution/final.pth


/home/ainshin/miniconda3/envs/ksw_pyg_dgl/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Sculptor Index: (6, 2, 0)
Rf scaler: mean=0.445887, std=0.232983


In [13]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")  # 파일에 BOM이 있음
df = df.reset_index(drop=True)
print(df.shape)
df.head()

(100, 5)


,Smiles,elu1_smiles,elu2_smiles,ratio1,ratio2
0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562
1,BrC/C=C\COC1CCCCO1,CCCCCC,CCOCC,0.650419,0.349581
2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197
3,BrCCCCCCNc1ccccc1,CCCCCC,CCOC(C)=O,0.789299,0.210701
4,BrCCc1cccc2ccccc12,CCCCCC,CCOC(C)=O,0.882244,0.117756


In [14]:
def score_batch(analyzer, batch):
    score = analyzer.get_score(
        Smiles=list(batch["Smiles"]),
        eluent1_smiles=list(batch["elu1_smiles"]),
        eluent2_smiles=list(batch["elu2_smiles"]),
        ratio1=[float(x) for x in batch["ratio1"]],
        ratio2=[float(x) for x in batch["ratio2"]],
    )
    y = np.asarray(score["y"], dtype=float).reshape(-1)
    contrib = np.asarray(score["subgroup_contributions"], dtype=float).reshape(-1)
    sizes = np.asarray(score["subgroup_size"]).reshape(-1).astype(int)

    offsets = np.concatenate([[0], np.cumsum(sizes)])
    return [(y[i], contrib[offsets[i]:offsets[i + 1]]) for i in range(len(batch))]


mol_rows = []
frag_rows = []

for start in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[start:start + BATCH_SIZE]
    results = score_batch(BLI_segment, batch)

    for (row_idx, row), (y_scaled, contribs) in zip(batch.iterrows(), results):
        fragments = BLI_segment.get_fragment(row["Smiles"])
        if len(fragments) != len(contribs):
            raise ValueError(
                f"row {row_idx}: fragment 수({len(fragments)}) != contribution 수({len(contribs)}) | {row['Smiles']}"
            )

        rf_pred = RF_MEAN + RF_STD * y_scaled
        mol_rows.append({"row_id": row_idx, "Rf_pred": rf_pred, "y_scaled": y_scaled, "n_fragments": len(fragments)})

        for frag_idx, (frag, c) in enumerate(zip(fragments, contribs)):
            frag_rows.append({
                "row_id": row_idx,
                "frag_idx": frag_idx,
                "frag_smiles": frag.smiles,
                "frag_atoms": " ".join(str(int(a)) for a in frag.atoms),
                "contribution": c,               # 
                # "contribution_Rf": RF_STD * c,  #
                "normalized_subgroup_contribution": np.tanh(c),  # [-1, 1]
            })

mol_df = pd.DataFrame(mol_rows)
frag_df = pd.DataFrame(frag_rows)
print(mol_df.shape, frag_df.shape)

Generating eluent2_smiles graphs: 100%|████████████████████████████████████████| 64/64 [00:00<00:00, 183.28it/s]


Generating eluent2_smiles graphs: 100%|████████████████████████████████████████| 36/36 [00:00<00:00, 182.47it/s]


(100, 4) (410, 7)


## save

In [17]:
out_df = (
    df.rename_axis("row_id").reset_index()
      .merge(mol_df, on="row_id")
      .merge(frag_df, on="row_id")
      .sort_values(["row_id", "frag_idx"])
)
out_df = out_df[[
    "row_id", "Smiles", "elu1_smiles", "elu2_smiles", "ratio1", "ratio2",
    "Rf_pred",
    "frag_idx", "frag_smiles", "frag_atoms",
     "normalized_subgroup_contribution", 
]]
out_df.to_csv(OUTPUT_CSV, index=False)
print("saved:", OUTPUT_CSV, out_df.shape)
out_df.head(10)

saved: ./data/Cpaper_sample_BLI_segment_sum_contribution_pred.csv (410, 11)


,row_id,Smiles,elu1_smiles,elu2_smiles,ratio1,ratio2,Rf_pred,frag_idx,frag_smiles,frag_atoms,normalized_subgroup_contribution
0,0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562,0.557744,0,*C=C(*)Br,1 2 0,0.161212
1,0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562,0.557744,1,c1ccccc1,9 10 11 12 13 14,0.080655
2,0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562,0.557744,2,C1CSCSC1,3 4 5 6 7 8,0.232325
3,1,BrC/C=C\COC1CCCCO1,CCCCCC,CCOCC,0.650419,0.349581,0.564704,0,OC/C=C\CBr,5 2 3 0 1 4,0.553084
4,1,BrC/C=C\COC1CCCCO1,CCCCCC,CCOCC,0.650419,0.349581,0.564704,1,C1CCOCC1,6 7 8 9 10 11,-0.112354
5,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,0,*Br,0,0.167024
6,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,1,*Br,3,0.175647
7,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,2,c1ccccc1,5 6 7 8 9 10,0.177695
8,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,3,C1=CCOC1,1 2 4 11 12,0.033906
9,3,BrCCCCCCNc1ccccc1,CCCCCC,CCOC(C)=O,0.789299,0.210701,0.445520,0,NCCCCCCBr,0 1 2 3 4 5 6 7,-0.075763


---
# `./data/Cpaper.csv`에 동일한 작업 적용

위의 과정(예측 → 범위 확인 → 검증 → 저장)을 함수 하나로 묶어서 `Cpaper.csv`에 실행합니다. `score_batch`, `RF_MEAN`, `RF_STD`는 위 셀에서 정의한 것을 그대로 사용합니다.
입력 CSV에 `Smiles`, `elu1_smiles`, `elu2_smiles`, `ratio1`, `ratio2` 외의 컬럼(예: 실측 `Rf`)이 있으면 결과 CSV에 함께 남깁니다.

In [18]:
def run_contribution_prediction(analyzer, input_csv, output_csv, batch_size=BATCH_SIZE):
    data = pd.read_csv(input_csv, encoding="utf-8-sig").reset_index(drop=True)
    print(f"input: {input_csv} {data.shape}")

    mol_rows, frag_rows = [], []
    for start in range(0, len(data), batch_size):
        batch = data.iloc[start:start + batch_size]
        results = score_batch(analyzer, batch)

        for (row_idx, row), (y_scaled, contribs) in zip(batch.iterrows(), results):
            fragments = analyzer.get_fragment(row["Smiles"])
            if len(fragments) != len(contribs):
                raise ValueError(
                    f"row {row_idx}: fragment 수({len(fragments)}) != contribution 수({len(contribs)}) | {row['Smiles']}"
                )

            mol_rows.append({
                "row_id": row_idx,
                "Rf_pred": RF_MEAN + RF_STD * y_scaled,
                "y_scaled": y_scaled,
                "n_fragments": len(fragments),
            })
            for frag_idx, (frag, c) in enumerate(zip(fragments, contribs)):
                frag_rows.append({
                    "row_id": row_idx,
                    "frag_idx": frag_idx,
                    "frag_smiles": frag.smiles,
                    "frag_atoms": " ".join(str(int(a)) for a in frag.atoms),
                    "contribution": c,
                    "normalized_subgroup_contribution": np.tanh(c),
                    "contribution_Rf": RF_STD * c,
                })

    mol_out = pd.DataFrame(mol_rows)
    frag_out = pd.DataFrame(frag_rows)

    # 범위 확인
    c = frag_out["contribution"]
    print(f"contribution range: {c.min():.4f} ~ {c.max():.4f}, |contribution| > 1 : {(c.abs() > 1).sum()} / {len(frag_out)}")
    print(f"normalized range  : {frag_out['normalized_subgroup_contribution'].min():.4f} ~ "
          f"{frag_out['normalized_subgroup_contribution'].max():.4f}")

    # 검증: Σ contribution == y_scaled
    chk = frag_out.groupby("row_id")["contribution"].sum().to_frame("sum_contrib").join(mol_out.set_index("row_id"))
    print("max |Σcontribution - y_scaled| =", np.abs(chk["sum_contrib"] - chk["y_scaled"]).max())

    input_cols = list(data.columns)
    result = (
        data.rename_axis("row_id").reset_index()
            .merge(mol_out, on="row_id")
            .merge(frag_out, on="row_id")
            .sort_values(["row_id", "frag_idx"])
    )
    result = result[
        ["row_id"] + input_cols
        + ["Rf_pred", "y_scaled", "n_fragments",
           "frag_idx", "frag_smiles", "frag_atoms",
           "contribution", "normalized_subgroup_contribution", "contribution_Rf"]
    ]
    result.to_csv(output_csv, index=False)
    print("saved:", output_csv, result.shape)
    return result

In [10]:
CPAPER_INPUT_CSV = "./data/Cpaper.csv"
CPAPER_OUTPUT_CSV = "./data/Cpaper_BLI_segment_sum_contribution_pred.csv"

cpaper_out = run_contribution_prediction(BLI_segment, CPAPER_INPUT_CSV, CPAPER_OUTPUT_CSV)
cpaper_out.head(10)

input: ./data/Cpaper.csv (100, 5)


Generating Smiles graphs:   0%|                                                                                                      | 0/64 [00:00<?, ?it/s]

Generating Smiles graphs:  22%|████████████████████▏                                                                       | 14/64 [00:00<00:00, 135.69it/s]

Generating Smiles graphs:  44%|████████████████████████████████████████▎                                                   | 28/64 [00:00<00:00, 130.04it/s]

Generating Smiles graphs:  66%|████████████████████████████████████████████████████████████▍                               | 42/64 [00:00<00:00, 125.74it/s]

Generating Smiles graphs:  86%|███████████████████████████████████████████████████████████████████████████████             | 55/64 [00:00<00:00, 125.50it/s]

... (18 outputs left out)

Generating eluent2_smiles graphs:  53%|████████████████████████████████████████████▎                                       | 19/36 [00:00<00:00, 186.19it/s]

Generating eluent2_smiles graphs: 100%|████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 179.07it/s]

contribution range: -1.1053 ~ 1.5100, |contribution| > 1 : 3 / 410
normalized range  : -0.8024 ~ 0.9069
max |Σcontribution - y_scaled| = 9.313225746154785e-08
saved: ./data/Cpaper_BLI_segment_sum_contribution_pred.csv (410, 15)


,row_id,Smiles,elu1_smiles,elu2_smiles,ratio1,ratio2,Rf_pred,y_scaled,n_fragments,frag_idx,frag_smiles,frag_atoms,contribution,normalized_subgroup_contribution,contribution_Rf
0,0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562,0.557744,0.480107,3,0,*C=C(*)Br,1 2 0,0.162631,0.161212,0.037890
1,0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562,0.557744,0.480107,3,1,c1ccccc1,9 10 11 12 13 14,0.080830,0.080655,0.018832
2,0,Br/C(=C\C1SCCCS1)c1ccccc1,CCCCCC,CCOC(C)=O,0.937438,0.062562,0.557744,0.480107,3,2,C1CSCSC1,3 4 5 6 7 8,0.236646,0.232325,0.055134
3,1,BrC/C=C\COC1CCCCO1,CCCCCC,CCOCC,0.650419,0.349581,0.564704,0.509984,2,0,OC/C=C\CBr,5 2 3 0 1 4,0.622814,0.553084,0.145105
4,1,BrC/C=C\COC1CCCCO1,CCCCCC,CCOCC,0.650419,0.349581,0.564704,0.509984,2,1,C1CCOCC1,6 7 8 9 10 11,-0.112830,-0.112354,-0.026287
5,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,0.559613,4,0,*Br,0,0.168604,0.167024,0.039282
6,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,0.559613,4,1,*Br,3,0.177488,0.175647,0.041352
7,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,0.559613,4,2,c1ccccc1,5 6 7 8 9 10,0.179602,0.177695,0.041844
8,2,BrC1=C(Br)C(c2ccccc2)OC1,CCCCCC,CCOC(C)=O,0.749803,0.250197,0.576267,0.559613,4,3,C1=CCOC1,1 2 4 11 12,0.033920,0.033906,0.007903
9,3,BrCCCCCCNc1ccccc1,CCCCCC,CCOC(C)=O,0.789299,0.210701,0.445520,-0.001574,2,0,NCCCCCCBr,0 1 2 3 4 5 6 7,-0.075908,-0.075763,-0.017685
